# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a practical workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll walk through the process of loading the Croissant schema, listing and extracting record sets using their `@id`, examining fields, performing exploratory analysis on the data, and visualizing key trends.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset Croissant schema and instantiate the `mlcroissant.Dataset` object. View high-level metadata on the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Set the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata, 'citeAs', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
List available record sets in the dataset and explore their structure. The `@id` field is used to uniquely identify each entity.

We first enumerate record sets, and then display example fields and columns for each.

In [ ]:
# List all record sets by @id
print("Available record sets in the dataset:")
record_sets = list(dataset.record_sets)
rs_info = []
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs['name']}")
    rs_info.append({'@id': rs['@id'], 'name': rs['name']})
    # Also print first few fields/columns:
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("    Fields/example columns:")
        for f in fields[:3]:
            print(f"      - field @id: {f['@id']}, name: {f.get('name', '')}")
    elif 'column' in rs:
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print("    Columns:")
        for c in columns[:3]:
            print(f"      - column @id: {c['@id']}, name: {c.get('name', '')}")
    else:
        print("    No fields/columns listed.")

# Show the structure for the first record set as an example
if record_sets:
    main_rs_id = record_sets[0]['@id']
    print(f"\nExample structure for record set @id='{main_rs_id}':")
    pprint.pprint(record_sets[0])
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
We extract records from each record set using their `@id`.

These records are loaded into pandas DataFrames for further analysis. Here we use the main table’s `@id` (as listed above) to load it.

In [ ]:
# Prepare a list of record set @id's to extract
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set '@id': {rs_id} with shape: {df.shape}")
    except Exception as e:
        print(f"Failed to load '{rs_id}': {e}")

# Choose the primary data record set for demonstration:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f"\nDataFrame columns for primary record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
We now conduct basic EDA on a numeric field in the main record set. This example demonstrates:
- Filtering rows where the field value is above a threshold
- Normalizing the numeric column
- Grouping by a suitable field for aggregation

**Replace the field `@id`s if working with a different table.**

In [ ]:
import numpy as np
# For this dataset, let's print the columns of the main record set and select common numeric and grouping fields interactively
df = dataframes[main_record_set_id]
print(f"Columns: {df.columns.tolist()}")

# Select one or two likely numeric fields and grouping fields by their actual column names (which are the @id of respective fields/columns)
numeric_field = None
for c in df.columns:
    # Typical protein data columns contain 'MW', 'abundance', or 'count' in the name
    if ('abund' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'peptide' in c.lower() or df[c].dtype in [np.float64, np.int64]):
        numeric_field = c
        break

if not numeric_field:
    print("Could not guess a numeric field.")
else:
    print(f"\nUsing '{numeric_field}' as numeric field for EDA.")

    # Threshold is set as 1/3 quantile for demo
    threshold = df[numeric_field].quantile(0.33) if np.issubdtype(df[numeric_field].dtype, np.number) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to guess a categorical/group field
    group_field = None
    for c in df.columns:
        if c != numeric_field and (('sample' in c.lower()) or ('accession' in c.lower()) or ('desc' in c.lower()) or (df[c].dtype == object)):
            group_field = c
            break
    if group_field:
        print(f"Grouping by {group_field}…\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean value of '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Plot the numeric data distributions and relationships between selected fields. These plots use the main numeric field and group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field:
    plt.figure(figsize=(7, 3))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

if group_field:
    plt.figure(figsize=(8, 4))
    # For clarity, plot top 10 groups by count
    top_groups = df[group_field].value_counts().nlargest(10).index
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
    plt.xticks(rotation=45)
    plt.title(f"'{numeric_field}' by '{group_field}' (top 10 groups)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load a Croissant-based dataset using `mlcroissant`, enumerate available record sets, and extract and analyze the main data table by referencing all entities using their `@id`. We performed common EDA steps and explored the distribution and grouping of key numeric fields relevant to mass spectrometry protein data. 

You can further build on this notebook to conduct in-depth statistical analyses, apply domain-specific filters, or integrate additional external information about proteins or samples.